# Bank Loan Default Prediction — Pipeline chống rò rỉ dữ liệu

Notebook xây dựng mô hình dự đoán khả năng khoản vay bị **Charged Off** từ thông tin đã biết tại thời điểm khoản vay được phê duyệt.

**Pipeline:** Nạp dữ liệu → Kiểm định chất lượng → Xác định nhãn và thời điểm dự đoán → Tạo đặc trưng → Chia Train/Validation/Test → EDA trên Train → Tiền xử lý trong Pipeline → Cross-validation → Hiệu chỉnh xác suất → Chọn ngưỡng → Đánh giá Test → Giải thích và giới hạn.

> Quy ước nhãn: `1 = Charged Off` (nợ xấu), `0 = Fully Paid` (đã trả hết). Các khoản `Current` chưa có kết quả cuối cùng nên không được gán thành `Fully Paid`.


## 1. Định nghĩa bài toán và phạm vi

- **Bài toán:** phân loại nhị phân có mất cân bằng lớp.
- **Thời điểm dự đoán:** sau khi các điều khoản vay (`loan_amount`, `term`, `int_rate`, `installment`, `sub_grade`) đã được xác định nhưng trước khi khoản vay bắt đầu phát sinh thanh toán.
- **Đối tượng huấn luyện:** chỉ các khoản vay đã có kết quả cuối cùng: `Fully Paid` hoặc `Charged Off`.
- **Mục tiêu sử dụng:** tạo điểm rủi ro hỗ trợ sàng lọc; không tự động phê duyệt/từ chối khoản vay.

### Các lỗi logic của notebook cũ đã được sửa

1. Không đổi `Current` thành `Fully Paid` vì đây là dữ liệu bị kiểm duyệt phải (right-censored).
2. Loại `total_payment`, `last_payment_date`, `next_payment_date`, `last_credit_pull_date` vì chúng phát sinh sau thời điểm cấp vay và gây **target leakage**.
3. Không dùng `id`, `member_id`; bỏ `application_type` vì chỉ có một giá trị; bỏ `emp_title` vì cardinality rất cao và khó tổng quát hóa.
4. Không xóa hàng loạt outlier theo IQR. Thu nhập cao vẫn có thể là quan sát hợp lệ; thay vào đó dùng biến đổi log và scaler trong pipeline.
5. Không mã hóa ordinal trước khi chia dữ liệu. Mọi imputation, scaling và one-hot encoding đều được fit bên trong từng fold.
6. Không chọn mô hình chỉ theo Accuracy; ưu tiên `PR-AUC`, đồng thời báo cáo Recall, Precision, F1, ROC-AUC và Balanced Accuracy.
7. Có `random_state`, baseline, cross-validation, validation để chọn ngưỡng và test độc lập để báo cáo cuối.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    PrecisionRecallDisplay,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
TARGET = "target_default"

## 2. Nạp dữ liệu và kiểm định cấu trúc


In [ ]:
# Tìm dữ liệu khi chạy từ thư mục gốc hoặc trực tiếp trong notebooks.
DATA_CANDIDATES = [
    Path('data/Bank Loan Dataset.csv'),
    Path('../data/Bank Loan Dataset.csv'),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Không tìm thấy data/Bank Loan Dataset.csv')

raw_df = pd.read_csv(DATA_PATH)
print(f'Nguồn dữ liệu: {DATA_PATH.resolve()}')
print(f'Kích thước dữ liệu gốc: {raw_df.shape[0]:,} dòng × {raw_df.shape[1]} cột')
display(raw_df.head())


In [ ]:
expected_columns = {
    "id", "member_id", "loan_status", "annual_income", "dti", "installment",
    "int_rate", "loan_amount", "total_acc", "emp_length", "term", "sub_grade",
    "home_ownership", "purpose", "verification_status", "application_type",
    "emp_title", "address_state", "issue_date", "last_credit_pull_date",
    "last_payment_date", "next_payment_date", "total_payment", "grade",
}
missing_columns = sorted(expected_columns - set(raw_df.columns))
if missing_columns:
    raise ValueError(f"CSV thiếu các cột bắt buộc: {missing_columns}")

quality_report = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing_n": raw_df.isna().sum(),
    "missing_pct": raw_df.isna().mean().mul(100),
    "n_unique": raw_df.nunique(dropna=False),
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

print(f"Dòng trùng hoàn toàn: {raw_df.duplicated().sum():,}")
print(f"ID trùng: {raw_df['id'].duplicated().sum():,}")
print(f"Member ID trùng: {raw_df['member_id'].duplicated().sum():,}")
display(quality_report)


In [ ]:
status_summary = (
    raw_df["loan_status"]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .reset_index(name="count")
)
status_summary["percentage"] = status_summary["count"] / len(raw_df)
display(status_summary.assign(percentage=status_summary["percentage"].map(lambda x: f"{x:.2%}")))

plt.figure(figsize=(7, 4))
ax = sns.barplot(data=status_summary, x="loan_status", y="count", color="#4C78A8")
ax.set(title="Phân bố trạng thái khoản vay trong dữ liệu gốc", xlabel="Trạng thái", ylabel="Số khoản vay")
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}")
plt.tight_layout()
plt.show()


## 3. Xác định nhãn và loại rò rỉ dữ liệu

`Current` không phải nhãn tốt: khoản vay vẫn đang hoạt động và có thể trở thành `Fully Paid` hoặc `Charged Off` trong tương lai. Notebook lưu số lượng bị loại để kiểm toán, nhưng không dùng các dòng này khi huấn luyện.

CSV gốc được giữ nguyên. Mọi lọc và biến đổi chỉ diễn ra trên bản sao trong bộ nhớ để pipeline có thể tái lập.


In [ ]:
final_statuses = ["Fully Paid", "Charged Off"]
model_source = raw_df.loc[raw_df["loan_status"].isin(final_statuses)].copy()
model_source[TARGET] = (model_source["loan_status"] == "Charged Off").astype("int8")

excluded_current = int((raw_df["loan_status"] == "Current").sum())
print(f"Loại {excluded_current:,} khoản Current vì chưa có kết quả cuối cùng.")
print(f"Còn lại {len(model_source):,} khoản vay đã hoàn tất.")
print(f"Tỷ lệ Charged Off: {model_source[TARGET].mean():.2%}")

leakage_columns = [
    "total_payment",
    "last_payment_date",
    "next_payment_date",
    "last_credit_pull_date",
]
identifier_or_unstable_columns = [
    "id", "member_id", "application_type", "emp_title", "issue_date",
    "loan_status", "grade", "address_state",
]

display(pd.DataFrame({
    "nhóm": ["Target leakage", "ID/hằng số/high-cardinality/không dùng"],
    "cột bị loại": [", ".join(leakage_columns), ", ".join(identifier_or_unstable_columns)],
}))


## 4. Tạo đặc trưng tại thời điểm cấp vay

Các đặc trưng mới đều được tính từ thông tin đã có trước khi phát sinh thanh toán:

- `emp_length_years`: số năm làm việc, với `< 1 year = 0`, `10+ years = 10`.
- `term_months`: kỳ hạn 36 hoặc 60 tháng.
- `log_annual_income`, `log_loan_amount`, `log_installment`: giảm độ lệch phải mà không xóa quan sát hợp lệ.
- `loan_to_income`: tỷ lệ số tiền vay / thu nhập năm.
- `annual_installment_to_income`: tổng tiền trả góp ước tính trong 12 tháng / thu nhập năm.

`address_state` được loại khỏi mô hình mặc định vì lợi ích dự báo nhỏ nhưng có nguy cơ trở thành proxy địa lý trong quyết định tín dụng. Nếu triển khai thực tế, cần bộ dữ liệu nhạy cảm phù hợp để đánh giá fairness theo quy định áp dụng.


In [ ]:
EMP_LENGTH_MAP = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10,
}


def build_application_features(df: pd.DataFrame) -> pd.DataFrame:
    # Tạo đặc trưng chỉ từ dữ liệu đã biết tại thời điểm cấp vay.
    features = pd.DataFrame(index=df.index)
    safe_income = df["annual_income"].clip(lower=1)

    features["dti"] = df["dti"]
    features["int_rate"] = df["int_rate"]
    features["total_acc"] = df["total_acc"]
    features["emp_length_years"] = df["emp_length"].map(EMP_LENGTH_MAP)
    features["term_months"] = pd.to_numeric(
        df["term"].astype(str).str.extract(r"(\d+)", expand=False), errors="coerce"
    )
    features["log_annual_income"] = np.log1p(df["annual_income"].clip(lower=0))
    features["log_loan_amount"] = np.log1p(df["loan_amount"].clip(lower=0))
    features["log_installment"] = np.log1p(df["installment"].clip(lower=0))
    features["loan_to_income"] = df["loan_amount"] / safe_income
    features["annual_installment_to_income"] = (12 * df["installment"]) / safe_income

    features["sub_grade"] = df["sub_grade"].astype("string")
    features["home_ownership"] = df["home_ownership"].astype("string")
    features["purpose"] = df["purpose"].astype("string")
    features["verification_status"] = df["verification_status"].astype("string")
    return features


X = build_application_features(model_source)
y = model_source[TARGET].copy()

assert len(X) == len(y)
assert not set(leakage_columns).intersection(X.columns)
assert set(y.unique()).issubset({0, 1})

print(f"Ma trận đặc trưng: {X.shape[0]:,} dòng × {X.shape[1]} cột đầu vào")
display(X.head())


## 5. Chia Train / Validation / Test trước khi EDA

- **Train 70%:** fit mô hình và các bước tiền xử lý.
- **Validation 15%:** chọn ngưỡng phân loại; không dùng để fit mô hình.
- **Test 15%:** chỉ mở một lần để báo cáo cuối.

`stratify=y` giữ tỷ lệ nợ xấu tương đương giữa ba tập. Với dữ liệu triển khai thật có nhiều giai đoạn, nên thay random split bằng **out-of-time split**.


In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE,
)

# 0.17647 × 85% ≈ 15% toàn bộ dữ liệu, để Train/Validation/Test ≈ 70/15/15.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.1764705882,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "Tập": ["Train", "Validation", "Test"],
    "Số dòng": [len(X_train), len(X_val), len(X_test)],
    "Tỷ lệ Charged Off": [y_train.mean(), y_val.mean(), y_test.mean()],
})
display(split_summary.assign(
    **{
        "Số dòng": split_summary["Số dòng"].map(lambda x: f"{x:,}"),
        "Tỷ lệ Charged Off": split_summary["Tỷ lệ Charged Off"].map(lambda x: f"{x:.2%}"),
    }
))


## 6. EDA chỉ trên tập Train


In [ ]:
train_eda = X_train.copy()
train_eda[TARGET] = y_train

grade_order = [f"{letter}{number}" for letter in "ABCDEFG" for number in range(1, 6)]
grade_rate = (
    train_eda.groupby("sub_grade", observed=False)[TARGET]
    .agg(default_rate="mean", count="size")
    .reindex(grade_order)
    .dropna()
    .reset_index()
)

term_rate = (
    train_eda.groupby("term_months", observed=False)[TARGET]
    .agg(default_rate="mean", count="size")
    .reset_index()
)

purpose_rate = (
    train_eda.groupby("purpose", observed=False)[TARGET]
    .agg(default_rate="mean", count="size")
    .query("count >= 100")
    .sort_values("default_rate", ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.barplot(data=grade_rate, x="sub_grade", y="default_rate", color="#E45756", ax=axes[0])
axes[0].set(title="Tỷ lệ Charged Off theo sub-grade", xlabel="Sub-grade", ylabel="Tỷ lệ nợ xấu")
axes[0].tick_params(axis="x", rotation=90)

sns.barplot(data=term_rate, x="term_months", y="default_rate", color="#4C78A8", ax=axes[1])
axes[1].set(title="Tỷ lệ Charged Off theo kỳ hạn", xlabel="Số tháng", ylabel="Tỷ lệ nợ xấu")

sns.barplot(data=purpose_rate, x="default_rate", y="purpose", color="#72B7B2", ax=axes[2])
axes[2].set(title="Tỷ lệ Charged Off theo mục đích (n ≥ 100)", xlabel="Tỷ lệ nợ xấu", ylabel="Mục đích")

for ax in axes:
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}")) if ax is not axes[2] else None
axes[2].xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()


In [ ]:
numeric_eda_cols = [
    "dti", "int_rate", "total_acc", "emp_length_years", "term_months",
    "log_annual_income", "log_loan_amount", "log_installment",
    "loan_to_income", "annual_installment_to_income",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
selected = [
    "dti", "int_rate", "log_annual_income", "log_loan_amount",
    "loan_to_income", "annual_installment_to_income",
]
for ax, column in zip(axes.flat, selected):
    sns.histplot(data=train_eda, x=column, hue=TARGET, bins=35, stat="density",
                 common_norm=False, element="step", alpha=0.25, ax=ax)
    ax.set_title(column)
plt.suptitle("Phân phối đặc trưng số trên tập Train", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 8))
sns.heatmap(train_eda[numeric_eda_cols].corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Tương quan Pearson giữa các đặc trưng số — Train")
plt.tight_layout()
plt.show()


## 7. Tiền xử lý bằng `ColumnTransformer`

- Biến số: điền median → chuẩn hóa StandardScaler.
- Biến phân loại: điền mode → One-Hot Encoding; giá trị mới ở dữ liệu tương lai được bỏ qua an toàn.
- Pipeline được đặt **bên trong cross-validation**, vì vậy encoder và scaler không nhìn thấy fold đánh giá.


In [ ]:
numeric_features = [
    "dti", "int_rate", "total_acc", "emp_length_years", "term_months",
    "log_annual_income", "log_loan_amount", "log_installment",
    "loan_to_income", "annual_installment_to_income",
]
categorical_features = [
    "sub_grade", "home_ownership", "purpose", "verification_status",
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print(f"Số đặc trưng số: {len(numeric_features)}")
print(f"Số đặc trưng phân loại gốc: {len(categorical_features)}")


## 8. So sánh mô hình bằng Stratified 5-Fold CV

- `DummyClassifier` là baseline bắt buộc.
- `LogisticRegression` là mô hình tuyến tính dễ giải thích, dùng `class_weight='balanced'`.
- `RandomForest` kiểm tra khả năng học quan hệ phi tuyến, đồng thời hạn chế overfit bằng `min_samples_leaf`.

KNN, Gaussian Naive Bayes và một cây Decision Tree đơn không được giữ trong pipeline chính: KNN kém phù hợp với dữ liệu one-hot nhiều chiều; GaussianNB giả định Gaussian và độc lập có điều kiện; cây đơn thường có phương sai cao. Có thể thêm chúng như thí nghiệm phụ nếu môn học yêu cầu.


In [ ]:
models = {
    "Dummy baseline": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(
        max_iter=2_000,
        solver="liblinear",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=8,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=1,
        random_state=RANDOM_STATE,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "PR_AUC": "average_precision",
    "ROC_AUC": "roc_auc",
    "F1": "f1",
    "Balanced_Accuracy": "balanced_accuracy",
}

cv_rows = []
for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", clone(estimator)),
    ])
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise",
    )
    row = {"Model": model_name}
    for metric_name in scoring:
        values = scores[f"test_{metric_name}"]
        row[f"{metric_name}_mean"] = values.mean()
        row[f"{metric_name}_std"] = values.std(ddof=1)
    cv_rows.append(row)

cv_summary = pd.DataFrame(cv_rows).sort_values("PR_AUC_mean", ascending=False).reset_index(drop=True)
display(cv_summary.round(4))


In [ ]:
plot_metrics = ["PR_AUC_mean", "ROC_AUC_mean", "F1_mean", "Balanced_Accuracy_mean"]
cv_long = cv_summary.melt(id_vars="Model", value_vars=plot_metrics,
                          var_name="Metric", value_name="Score")
cv_long["Metric"] = cv_long["Metric"].str.replace("_mean", "", regex=False)

plt.figure(figsize=(11, 5))
sns.barplot(data=cv_long, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1)
plt.title("So sánh mô hình trên Train bằng Stratified 5-Fold CV")
plt.ylabel("Điểm trung bình")
plt.tight_layout()
plt.show()


## 9. Chọn mô hình, hiệu chỉnh xác suất và chọn ngưỡng trên Validation

Mô hình tốt nhất được chọn theo **PR-AUC trung bình** vì lớp `Charged Off` là lớp thiểu số. Sau đó:

1. Fit lại trên toàn bộ Train.
2. Hiệu chỉnh xác suất bằng sigmoid calibration với 3-fold nội bộ trên Train.
3. Chọn ngưỡng tối đa F1 trên Validation.

Ngưỡng tối đa F1 chỉ là mặc định trung lập. Khi triển khai, ngân hàng cần thay bằng hàm chi phí phản ánh tổn thất do bỏ sót nợ xấu và chi phí từ chối nhầm khách tốt.


In [ ]:
candidate_summary = cv_summary[cv_summary["Model"] != "Dummy baseline"]
best_model_name = candidate_summary.iloc[0]["Model"]

best_base_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", clone(models[best_model_name])),
])

# CalibratedClassifierCV tự fit pipeline trong từng fold, nên preprocessing vẫn leak-free.
try:
    calibrated_model = CalibratedClassifierCV(
        estimator=best_base_pipeline, method="sigmoid", cv=3, n_jobs=-1
    )
except TypeError:  # Tương thích với scikit-learn cũ.
    calibrated_model = CalibratedClassifierCV(
        base_estimator=best_base_pipeline, method="sigmoid", cv=3, n_jobs=-1
    )

calibrated_model.fit(X_train, y_train)
val_proba = calibrated_model.predict_proba(X_val)[:, 1]

pr_precision, pr_recall, thresholds = precision_recall_curve(y_val, val_proba)
f1_by_threshold = (
    2 * pr_precision[:-1] * pr_recall[:-1]
    / (pr_precision[:-1] + pr_recall[:-1] + 1e-12)
)
best_idx = int(np.nanargmax(f1_by_threshold))
best_threshold = float(thresholds[best_idx])

print(f"Mô hình được chọn theo CV PR-AUC: {best_model_name}")
print(f"Ngưỡng tối đa F1 trên Validation: {best_threshold:.4f}")
print(f"Validation PR-AUC: {average_precision_score(y_val, val_proba):.4f}")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, val_proba):.4f}")
print(f"Validation F1 tại ngưỡng đã chọn: {f1_by_threshold[best_idx]:.4f}")


In [ ]:
def threshold_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "flagged_n": int(predictions.sum()),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, predictions, zero_division=0),
    }


scenario_thresholds = sorted(set([0.10, 0.15, 0.20, 0.25, 0.30, best_threshold, 0.40, 0.50]))
threshold_table = pd.DataFrame([
    threshold_metrics(y_val, val_proba, threshold)
    for threshold in scenario_thresholds
])
display(threshold_table.round(4))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, pr_precision[:-1], label="Precision")
plt.plot(thresholds, pr_recall[:-1], label="Recall")
plt.plot(thresholds, f1_by_threshold, label="F1")
plt.axvline(best_threshold, color="black", linestyle="--", label=f"Ngưỡng chọn = {best_threshold:.3f}")
plt.xlabel("Ngưỡng xác suất")
plt.ylabel("Điểm")
plt.title("Chọn ngưỡng trên tập Validation")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Đánh giá cuối cùng trên Test độc lập


In [ ]:
test_proba = calibrated_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

test_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy", "Balanced Accuracy", "Precision (Charged Off)",
        "Recall (Charged Off)", "F1 (Charged Off)", "ROC-AUC", "PR-AUC", "Brier Score",
    ],
    "Value": [
        accuracy_score(y_test, test_pred),
        balanced_accuracy_score(y_test, test_pred),
        precision_score(y_test, test_pred, zero_division=0),
        recall_score(y_test, test_pred, zero_division=0),
        f1_score(y_test, test_pred, zero_division=0),
        roc_auc_score(y_test, test_proba),
        average_precision_score(y_test, test_proba),
        brier_score_loss(y_test, test_proba),
    ],
})
display(test_metrics.round(4))

print("Classification report — Test")
print(classification_report(
    y_test,
    test_pred,
    target_names=["Fully Paid", "Charged Off"],
    digits=4,
))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred,
    display_labels=["Fully Paid", "Charged Off"],
    cmap="Blues", values_format=",d", ax=axes[0, 0],
)
axes[0, 0].set_title(f"Confusion matrix — threshold {best_threshold:.3f}")

PrecisionRecallDisplay.from_predictions(y_test, test_proba, ax=axes[0, 1], name=best_model_name)
axes[0, 1].axhline(y_test.mean(), color="gray", linestyle="--", label="Tỷ lệ lớp dương")
axes[0, 1].set_title("Precision–Recall curve — Test")
axes[0, 1].legend()

RocCurveDisplay.from_predictions(y_test, test_proba, ax=axes[1, 0], name=best_model_name)
axes[1, 0].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1, 0].set_title("ROC curve — Test")

CalibrationDisplay.from_predictions(
    y_test, test_proba, n_bins=10, strategy="quantile", ax=axes[1, 1], name=best_model_name
)
axes[1, 1].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1, 1].set_title("Calibration curve — Test")

plt.tight_layout()
plt.show()


## 11. Giải thích mô hình ở mức đặc trưng gốc

Permutation importance đo mức giảm PR-AUC khi xáo trộn từng cột đầu vào trên Test. Cách này hoạt động thống nhất cho cả Logistic Regression và Random Forest, đồng thời gom toàn bộ dummy one-hot về cột gốc dễ hiểu.

> Importance cho biết mô hình phụ thuộc vào đặc trưng nào, không chứng minh quan hệ nhân quả và không cho biết chiều tác động.


In [ ]:
permutation = permutation_importance(
    calibrated_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
display(importance_df.round(5))

plot_imp = importance_df.sort_values("importance_mean", ascending=True)
plt.figure(figsize=(9, 6))
plt.barh(
    plot_imp["feature"],
    plot_imp["importance_mean"],
    xerr=plot_imp["importance_std"],
    color="#4C78A8",
    alpha=0.85,
)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Mức giảm PR-AUC khi xáo trộn")
plt.ylabel("Đặc trưng")
plt.title("Permutation importance trên Test")
plt.tight_layout()
plt.show()


## 12. Kiểm tra tính hợp lệ và kết luận

### Những gì kết quả này chứng minh

- Pipeline vượt baseline nếu PR-AUC của mô hình cao hơn tỷ lệ `Charged Off` và CV ổn định giữa các fold.
- Test được giữ độc lập khỏi việc fit mô hình và chọn ngưỡng, nên phản ánh trung thực hơn khả năng tổng quát hóa nội bộ.
- Precision và Recall phải được đọc cùng nhau. Với dữ liệu mất cân bằng, Accuracy cao không đồng nghĩa phát hiện nợ xấu tốt.

### Chưa đủ tiêu chuẩn triển khai ngân hàng

1. Dataset không có mô tả nguồn, thời điểm quan sát và quy tắc tạo nhãn đầy đủ; các ngày đều tập trung trong một năm nên chưa thể đánh giá drift theo thời gian.
2. Chưa có dữ liệu nhạy cảm cần thiết để kiểm tra fairness đầy đủ; tuyệt đối không suy diễn fairness từ việc bỏ `address_state`.
3. Ngưỡng F1 chưa phản ánh chi phí kinh doanh, LGD/EAD hoặc lợi nhuận kỳ vọng.
4. Cần out-of-time validation, external validation, stress test, kiểm tra calibration theo nhóm và giám sát drift trước khi dùng thực tế.
5. Nếu cần giữ cả khoản `Current`, phải chuyển sang survival analysis / time-to-event thay vì gán nhãn giả.

### Hướng phát triển portfolio

- Thêm MLflow/DVC để theo dõi thí nghiệm và phiên bản dữ liệu.
- Viết unit test cho schema, feature engineering và leakage guard.
- Đóng gói pipeline bằng `joblib`, xây API FastAPI và giao diện demo.
- Thêm cost-sensitive threshold, expected loss và model card.


In [ ]:
# Tóm tắt có thể dùng trực tiếp trong báo cáo.
metric_lookup = test_metrics.set_index("Metric")["Value"]
print(
    f"Mô hình {best_model_name} được chọn theo PR-AUC trung bình trên Stratified 5-Fold CV. "
    f"Sau hiệu chỉnh xác suất và chọn ngưỡng {best_threshold:.3f} trên Validation, "
    f"mô hình đạt PR-AUC={metric_lookup['PR-AUC']:.4f}, "
    f"ROC-AUC={metric_lookup['ROC-AUC']:.4f}, "
    f"Recall={metric_lookup['Recall (Charged Off)']:.4f} và "
    f"F1={metric_lookup['F1 (Charged Off)']:.4f} trên Test độc lập."
)

# Khi cần lưu mô hình sau khi đã chốt pipeline:
# import joblib
# joblib.dump({"model": calibrated_model, "threshold": best_threshold}, "bank_loan_model.joblib")
